# Dataset Construction
----

**Updated:** *January 2026*

This notebook contains the code used for dataset construction.

# Table of Contents
  - [1. Constructing Product Level Dataset](#1-constructing-product-level-dataset)
    - [1.1. Load Dataset](#11-load-dataset)
    - [1.2. Construct Variables to be Collapsed](#12-construct-variables-to-be-collapsed)
    - [1.3. Collapse Data Down to Project Level](#13-collapse-data-down-to-project-level)
  - [2. Individual Level Variable Construction](#20-individual-level-variable-construction)
    - [2.1. Construct Position Level Variables](#21-construct-position-level-variables)
    - [2.2. Construct Variables for Technology Diffusion and Mobility](#22-construct-variables-for-technology-diffusion-and-mobility)
  - [3. Output Data for Analysis](#30-Output-Data-For-Analysis)

In [1]:
// Preamble //
clear all
set more off
set linesize 255


## 1. Constructing Product Level Dataset
-----

In this Section of the notebook, we are loading the code at the product / project level and cleaning it. Mobility is based on movement between projects and therefore the use of middelware components, team size and product sales are all based on product level measures. Here we demonstrate the code used to generate those measures. 

### 1.1. Load Dataset 
----

Note, data is at the project (game) platform level and is therefore repeated for multiple platforms. To address this, we are constructing and cleaning variables at thh game level and then collapsing the data down to one observation per game.

In [2]:
*##############################################################################*
* LOAD MOBY GAMES DATASET 
*##############################################################################*

use "./Data/Project Level Data - MISQ - Middleware 2025.dta", clear

In [3]:
*##############################################################################*
* CLEAN UP RELEASE DATES 
*##############################################################################*

qui: gen release_date = reldate
qui: gen release_year = year(release_date)
qui: drop if release_year < 1990

qui: gen date = release_date - date("01Jan1990", "DMY")

*##############################################################################*
* REMOVE GAMES ON PC AND WINDOWS PLATFORMS 
*##############################################################################*

qui: decode(platform), generate(Ptfm)
qui: drop if Ptfm == "DOS" | Ptfm == "Windows" | Ptfm == "Windows 3.x" | Ptfm == "Commodore 64" | Ptfm == "PC Booter"

#### 1.2. Construct Variables to be Collapsed 
----

Here we construct indicator variables that represent each product. These will be collapsed down to the product level. There are often multiple rows per product because a single game may be availible on different platforms. This most impacts the platform indicators and the product sales variables which are different for each platform. The other columns are the same for all releases of a single title. 

In [4]:
// CREATE DUMMY VARIABLES FOR EACH PLATFORM //
qui: tabulate platform, generate(Ptfm_)

// IDENTIFY GAMES WHICH ARE IN THE SHOOTER GENRE //
qui: gen SHOOTER = (npd_supergenre == "SHOOTER")

// IDENTIFY GAMES THAT USE AT LEAST ONE MIDDLEWARE COMPONENT 
qui: egen MIDDLE = rmax(GameEngine_3rdparty GraphicsEngine_3rdparty ThreedEngine_3rdparty PhysicsEngine)
qui: egen INHOUSE = rmax(GameEngine_inhouse GraphicsEngine_inhouse ThreedEngine_inhouse)

// IDENTIFY DIFFERENT MIDDLEWARE COMPONENTS INCLUDING (MOST POPULAR) COMPONENTS//
qui: gen E_Graphics = "0.NONE" if GraphicsEngine_name1 == ""
qui: replace E_Graphics = "1.Less Popular" if GraphicsEngine_name1!="RenderWare" & GraphicsEngine_name1!=""
qui: replace E_Graphics = "2.Popular" if GraphicsEngine_name1=="RenderWare" 

qui: gen E_3D = "0.None" if ThreedEngine_name1 == ""
qui: replace E_3D = "1.Less Popular" if !regexm(ThreedEngine_name1, "Unreal") & ThreedEngine_name1!=""
qui: replace E_3D = "2.Popular" if regexm(ThreedEngine_name1, "Unreal") 

qui: gen E_Phys = "0.None" if PhysicsEngine_name1 == ""
qui: replace E_Phys = "1.Less Popular" if !regexm(PhysicsEngine_name1, "Havok") & PhysicsEngine_name1!=""
qui: replace E_Phys = "2.Popular" if regexm(PhysicsEngine_name1, "Havok") 

// IDENTIFY GAMES THAT USE AT LEAST ONE MAJOR | POPULAR MIDDLEWARE COMPONENT //
qui: gen PopularMiddleware = 1 if regexm(E_3D, "2.") | regexm(E_Graphics, "2.") | regexm(E_Phys, "2.")
qui: replace PopularMiddleware = 0 if PopularMiddleware == .

// BECAUSE RENDERWARE WAS NO LONGER THIRD PARTY AFTER BEING PURCHASED BY EA, WE ENSURE IT IS ALWAYS CONSIDERED THIRD PARTY //
qui: replace MIDDLE=1 if PopularMiddleware==1

#### 1.3. Collapse Data Down to Project Level 
-----

In [5]:
*##############################################################################*
* SORT THE DATA PRIOR TO COLLAPSING
* ---------------------------------------------------------------------------- *
* NOTE THAT SORTING IS DONE TO ENSURE THAT THE PLATFORMS ARE ALWAYS LISTED IN THE SAME ORDER 
* SUCH THAT THE STUDIO FOR A TITLE IS ALWAYS LISTED -- THIS IS IMPORTANT BECAUSE 
* DIFFERENT PLATFORMS MAY HAVE DIFFERENT DEVELOPERS (OR DEVELOPR NAMES)
* FOR THE SAME GAME, AND THIS ENSURES THAT THE FIRST OBSERVATION IS ALWAYS THE SAME.  
*##############################################################################*

gsort urlnew -npd_supergenre
qui: by urlnew: replace npd_supergenre=npd_supergenre[_n-1] if npd_supergenre[_n-1]!=""

sort urlnew release_date Ptfm MIDDLE

// COLLAPSE THE DATA TO ONE OBSERVATION PER GAME //
#delimit; 

collapse (firstnm) title publisher pub_corp dev_corp developer game_id npd_supergenre npd_genre E_* 
         (max) SHOOTER LicensedTitle inhouse PopularMiddleware* Ptfm_* MIDDLE INHOUSE   
         (sum) CumulativeSales 
         (min) release_date date release_year, 
         by(urlnew);

#delimit cr

// LOG TRANSFORM TOTAL SALES VARIABLE AFTER IT HAS BEEN AGGREGATED //
qui: gen logRevenue = ln(CumulativeSales)

In [6]:
save "./Data/project level data.dta", replace

file ./Data/project level data.dta saved


## 2. Individual Level Variable Construction  
----

Worker mobility is based on the individual level which contains information about each worker, their mobility between projects. In this stage, the two datasets are combined: Project Level Data which Contains Information about each title and middleware use & Individual Level data which contains information about each worker and their experience. 

In [7]:
*##############################################################################*
* LOAD PROJECT LEVEL DATA 
*###############################################################################*

use "./Data/project level data.dta", clear 

*##############################################################################*
* MERGE IN INDIVIDUAL LEVEL DATA 
*###############################################################################*

qui: joinby urlnew using "./Data/Individual Level Data - MISQ - Middleware 2025.dta"

*##############################################################################*
* MERGE IN SEX AND ETHNICITY DATA (FROM CLASSIFIER BASED ON INDIVIDUAL NAMES)
*###############################################################################*

qui: merge m:1 developer_id using "./Data/classified names.dta"

// CLEAN UP GENDER AND ETHNICITY DATA //
qui: replace gender = "unknown" if gender == "andy"
qui: replace gender = "female" if gender == "mostly_female"
qui: replace gender = "male" if gender == "mostly_male"

qui: egen GenderID = group(gender)

qui: gen ethnicity = race
qui: egen EthnicityID = group(ethnicity)

qui: drop gender ethnicity race 


### 2.1. Construct Position Level Variables 
-----

In [8]:
*##############################################################################*
* CLEAN UP INDIVIDUAL LEVEL DATA AND CONSTRUCT VARIABLES
*###############################################################################*

// REMOVE WORKERS FOR WHO WE DO NOT HAVE INFORMATION ABOUT WHEN THE TITLE WAS RELEASED //
qui: drop if missing(date)

// CONSTRUCT WORKER UNIQUE ID //
qui: egen ID = group(developer_id) 

In [9]:
*###############################################################################*
* CLEAN UP TEXTUAL DATA FOR VARIABLES 
*###############################################################################*

// CLEAN DOUBLE QUOTES WITHIN SOME ROLE NAMES
qui: replace ruo_role = strtrim(ruo_role)
qui: replace ruo_role = ustrregexra(ruo_role, `"[\"""]"', "")

// DEFINE WORKER ROLES //
qui: replace ruo_role = lower(ruo_role)

qui: gen Hierarchy = "5.Other"
qui: replace Hierarchy = "1.Executive" if regexm(ruo_role, "game director") | regexm(ruo_role, "producer|production")
qui: replace Hierarchy = "2.Programmer" if regexm(ruo_department, "Programming|Engineering") 
qui: replace Hierarchy = "3.Creative" if (regexm(ruo_department, "Design|Art|Graphics|Writers|Audio|Video")|regexm(ruo_role, "audio|sound|music")) | (regexm(ruo_role,"graphics") & Hierarchy !="2.Programmer") 
qui: replace Hierarchy = "4.Testers" if regexm(ruo_role, "tester")

qui: egen HID = group(Hierarchy)

// DISTINGUISH HIGH AND LOW LEVEL ROLES //
qui: gen lowlevelprogrammer=0
qui: replace lowlevelprogrammer =1 if HID == 2 & (strmatch(ruo_role,"*engine ")==1 | strmatch(ruo_role,"*tool*")==1 | strmatch(ruo_role,"*graphic*")==1 |strmatch(ruo_role,"*library*")==1 | strmatch(ruo_role,"*low level*")==1 | strmatch(ruo_role,"*optimiz*")==1 | strmatch(ruo_role,"*render*")==1 | strmatch(ruo_role,"*shader*")==1 | strmatch(ruo_role,"*r&d*")==1 | strmatch(ruo_role,"*r'n'd*")==1 | strmatch(ruo_role,"*system*")==1 | strmatch(ruo_role,"*physics*")==1 | strmatch(ruo_role,"*simulation*")==1 | strmatch(ruo_role,"*advanced techno*")==1 | strmatch(ruo_role,"*core techno*")==1 | strmatch(ruo_role,"*platform*")==1 | strmatch(ruo_role,"ps3*")==1 | strmatch(ruo_role,"*architect*")==1 | strmatch(ruo_role,"*conversion*")==1 | strmatch(ruo_role,"asura *")==1 | strmatch(ruo_role,"console*")==1 | strmatch(ruo_role,"converter*")==1 | strmatch(ruo_role,"*framework*")==1 | strmatch(ruo_role,"ngl *")==1 | strmatch(ruo_role,"*architect*")==1 | strmatch(ruo_role,"xbox *")==1 | strmatch(ruo_role,"x360 *")==1 | strmatch(ruo_role,"xbox360 *")==1 | strmatch(ruo_role,"*abstraction*")==1 | strmatch(ruo_role,"dreamcast *")==1 | strmatch(ruo_role,"lithtech *")==1 | strmatch(ruo_role,"gamecube *")==1 | strmatch(ruo_role,"playstation*")==1 | strmatch(ruo_role,"ps2*")==1 | strmatch(ruo_role,"ps3*")==1 | strmatch(ruo_role,"psx*")==1 | strmatch(ruo_role,"multiplatform*")==1 | strmatch(ruo_role,"multi-platform*")==1 | strmatch(ruo_role,"*microcode*")==1 | strmatch(ruo_role,"vu *")==1 | strmatch(ruo_role,"vu1 *")==1 | strmatch(ruo_role,"*dynamics*")==1)
qui: replace lowlevelprogrammer =1 if HID == 2 & (regexm(ruo_department, "Technology") & regexm(ruo_role,"tools|technology|technical|technologies|control tech|technologist|tech|central technology|digital engineer|r&d|r & d|research and development|engine|abstraction")) & regexm(ruo_role,"online|web|information")==0
qui: replace lowlevelprogrammer = 1 if strmatch(ruo_role, "system engineer") | strmatch(ruo_role, "technology group director") | strmatch(ruo_role, "technical coordinators") | strmatch(ruo_role, "integration engineers") | strmatch(ruo_role, "integration engineering team") | strmatch(ruo_role, "effect programmers") | strmatch(ruo_role, "effect programmer") | strmatch(ruo_role, "yeti engine") | strmatch(ruo_role, "game engine team") | strmatch(ruo_role, "technology director") | strmatch(ruo_role, "character technical director") | strmatch(ruo_role, "effect program") | strmatch(ruo_role, "senior standards technicians") | strmatch(ruo_role, "technical leads") | strmatch(ruo_role, "mechanical engineer") | strmatch(ruo_role, "technology team") | strmatch(ruo_role, "pure 3d") | strmatch(ruo_role, "technical team") | strmatch(ruo_role, "floor leads, technical requirements group") | strmatch(ruo_role, "pure3d") | strmatch(ruo_role, "engine") | strmatch(ruo_role, "additional engine programming") | strmatch(ruo_role, "technical engineers") | strmatch(ruo_role, "integration") | strmatch(ruo_role, "technology programmers") | strmatch(ruo_role, "technical board") | strmatch(ruo_role, "integrators") | strmatch(ruo_role, "technical dept.") | strmatch(ruo_role, "technical standard analysts") | strmatch(ruo_role, "technology programming team") | strmatch(ruo_role, "technology group") | strmatch(ruo_role, "technical standards analyst") | strmatch(ruo_role, "technical supervisor") | strmatch(ruo_role, "3d engine") | strmatch(ruo_role, "frostbite engine team") | strmatch(ruo_role, "hardware engineer") | strmatch(ruo_role, "engine technology") | strmatch(ruo_role, "central technology") | strmatch(ruo_role, "engine programmer") | strmatch(ruo_role, "infrastructure") | strmatch(ruo_role, "technology programming") | strmatch(ruo_role, "senior engine programmers") | strmatch(ruo_role, "libraries / utilities") | strmatch(ruo_role, "lead engine programmer") | strmatch(ruo_role, "technical direction") | strmatch(ruo_role, "technical producer") | strmatch(ruo_role, "engine team") | strmatch(ruo_role, "3d programmers") | strmatch(ruo_role, "technical lead") | strmatch(ruo_role, "technology") | strmatch(ruo_role, "director of technology") | strmatch(ruo_role, "technical standards analysts") | strmatch(ruo_role, "technical directors") | strmatch(ruo_role, "engine programming") | strmatch(ruo_role, "engine programmers")
qui: replace lowlevelprogrammer = 0 if (regexm(ruo_role, "engineer")==1 & regexm(ruo_role, "tool|systems|r&d|library|platform|lithtech|ps2|physics|[[:<:]]engine[[:>:]]|digital")==0)

qui: gen lowlevelcreative=0
qui: replace lowlevelcreative = 1 if HID == 3 & (strmatch(ruo_role,"*sound engineer*") | strmatch(ruo_role,"*foley engineer*") | strmatch(ruo_role,"*sound design*") | strmatch(ruo_role,"*sound mixer*") | strmatch(ruo_role,"*sound system*") | strmatch(ruo_role,"*sound. tools*") | strmatch(ruo_role,"*video engineer*") | strmatch(ruo_role,"*video capture*") | strmatch(ruo_role,"*video compression*") | strmatch(ruo_role,"*video conversion*") | strmatch(ruo_role,"*post fx*") | strmatch(ruo_role,"*post-fx*") | strmatch(ruo_role,"*post production*") | strmatch(ruo_role,"*post-production*") | strmatch(ruo_role,"*post-processing*") | strmatch(ruo_role,"*remastering*") | strmatch(ruo_role,"*re-master*") | strmatch(ruo_role,"*encoding*") | strmatch(ruo_role,"*mastering*") | strmatch(ruo_role,"*movie audio editing*") | strmatch(ruo_role,"*newscuts*") | strmatch(ruo_role,"*protools*") | strmatch(ruo_role,"*vfx*") | strmatch(ruo_role,"*visual effect*") | strmatch(ruo_role,"*visual fx*") | strmatch(ruo_role,"*special effects*") | strmatch(ruo_role,"*effects*") | strmatch(ruo_role,"*effect animator*") | strmatch(ruo_role,"*effect animation*") | strmatch(ruo_role,"*effect design*") | strmatch(ruo_role,"*efx*") | strmatch(ruo_role,"*fx programmer*") | strmatch(ruo_role,"*fx cg*") | strmatch(ruo_role,"*scanner*") | strmatch(ruo_role,"*scanning*") | strmatch(ruo_role,"*capture*") | strmatch(ruo_role,"*mocap*") | strmatch(ruo_role,"*mo cap*") | strmatch(ruo_role,"*mo-cap*") | strmatch(ruo_role,"*motion capture*") | strmatch(ruo_role,"*facial capture*") | strmatch(ruo_role,"*game capture*") | strmatch(ruo_role,"*texture*") | strmatch(ruo_role,"*textur*") | strmatch(ruo_role,"*shader*") | strmatch(ruo_role,"*shading*") | strmatch(ruo_role,"*shade*") | strmatch(ruo_role,"*render*") | strmatch(ruo_role,"*renders*") | strmatch(ruo_role,"*graphics engineering*") | strmatch(ruo_role,"*graphics system*") | strmatch(ruo_role,"*graphic concepts*") | strmatch(ruo_role,"*graphical enhancer*") | strmatch(ruo_role,"*game graphics*") | strmatch(ruo_role,"*height map*") | strmatch(ruo_role,"*shadow*") | strmatch(ruo_role,"*user interface*") | strmatch(ruo_role,"*interface*") | strmatch(ruo_role,"*ui*") | strmatch(ruo_role,"*user experience*") | strmatch(ruo_role,"*ux*") | strmatch(ruo_role,"*front end*") | strmatch(ruo_role,"*front-end*") | strmatch(ruo_role,"*hud*") | strmatch(ruo_role,"*compositing*") | strmatch(ruo_role,"*compositor*") | strmatch(ruo_role,"*composit*") | strmatch(ruo_role,"*layout*") | strmatch(ruo_role,"*assembly*") | strmatch(ruo_role,"*interior assembly*") | strmatch(ruo_role,"*level assembly*") | strmatch(ruo_role,"*breakable models*") | strmatch(ruo_role,"*pipeline*") | strmatch(ruo_role,"*framework*") | strmatch(ruo_role,"*framing*") | strmatch(ruo_role,"*scripting*") | strmatch(ruo_role,"*script coder*") | strmatch(ruo_role,"*scriptor*") | strmatch(ruo_role,"*scripter*") | strmatch(ruo_role,"*sdk*") | strmatch(ruo_role,"*code*") | strmatch(ruo_role,"*program*") | strmatch(ruo_role,"*software design*") | strmatch(ruo_role,"*software*") | strmatch(ruo_role,"*implementation*") | strmatch(ruo_role,"*technical*") | strmatch(ruo_role,"*technology*") | strmatch(ruo_role,"*tech*") | strmatch(ruo_role,"*techs*") | strmatch(ruo_role,"*setup*") | strmatch(ruo_role,"*set-up*") | strmatch(ruo_role,"*set up*") | strmatch(ruo_role,"*tools*") | strmatch(ruo_role,"*tool*") | strmatch(ruo_role,"*convert*") | strmatch(ruo_role,"*conversion*") | strmatch(ruo_role,"*process*") | strmatch(ruo_role,"*library*") | strmatch(ruo_role,"*art td*") | strmatch(ruo_role,"*character td*") | strmatch(ruo_role,"*wrangler*") | strmatch(ruo_role,"*integrat*") | strmatch(ruo_role,"*research*") | strmatch(ruo_role,"*rigger*") | strmatch(ruo_role,"*rigging*") | strmatch(ruo_role,"*tweaker*") | strmatch(ruo_role,"*lighting*") | strmatch(ruo_role,"*fx*") | strmatch(ruo_role,"*mesh*") | strmatch(ruo_role,"*event editor*") | strmatch(ruo_role,"*object engineer*") | strmatch(ruo_role,"*control system*") | strmatch(ruo_role,"*dynamics*") | strmatch(ruo_role,"*game logic*") | strmatch(ruo_role,"*game shell*") | strmatch(ruo_role,"*in-game camera*") | strmatch(ruo_role,"*modeler*") | strmatch(ruo_role,"*model er*") | strmatch(ruo_role,"*modeling*") | strmatch(ruo_role,"*maya operator*") | strmatch(ruo_role,"*mdel production*") | strmatch(ruo_role,"*motion programmer*") | strmatch(ruo_role,"*motion editor*") | strmatch(ruo_role,"*motion simulation*") | strmatch(ruo_role,"*interactive music programming*") | strmatch(ruo_role,"*music logic*") | strmatch(ruo_role,"*synchronization*"))

### 2.2. Construct Variables for Technology Diffusion and Mobility 
----

In [10]:
// CONSTRUCT COUNTER OF JOB NUMBER //

gsort ID date urlnew -HID -lowlevelprogrammer -lowlevelcreative
qui: egen id_urlnew_tag=tag(ID urlnew)
qui: drop if id_urlnew_tag==0
qui: by ID: gen JobN=sum(id_urlnew_tag)

// CONTRUCT MOBILITY VARIABLE (CHANGING BETWEEN DEVELOPER COMPANIES) //
gsort ID date urlnew
qui: by ID: gen DevChange = 1 if developer[_n] != developer[_n-1]
qui: replace DevChange = 0 if DevChange == .
qui: replace DevChange = . if JobN == 1
qui: replace DevChange = . if id_urlnew_tag==0

// CONTRUCT VARIABLE INDICATING WHEN BOTH COMPANIES USE MIDDLEWARE //
gsort ID date urlnew
qui: by ID:  gen MiddlewareChange = 1 if MIDDLE[_n] != MIDDLE[_n-1]
qui: replace MiddlewareChange = 0 if MiddlewareChange == .
qui: replace MiddlewareChange = . if JobN == 1

gsort ID date urlnew
qui: by ID: gen NonMiddlewareToMiddleware = 1 if MIDDLE[_n] ==1 & MIDDLE[_n-1] != 1
qui: replace NonMiddlewareToMiddleware = 0 if NonMiddlewareToMiddleware == .
qui: replace NonMiddlewareToMiddleware = . if JobN == 1

gsort ID date urlnew
qui: by ID:  gen MIDDLEP = MIDDLE[_n-1]

// IDENTIFY ROLE IN PREVIOUS JOB // 
gsort ID date urlnew
qui: by ID: gen HID_Previous = HID[_n-1] if _n > 1
qui: replace HID_Previous = . if HID_Previous == 0

// CONSTRUCT CONTROLS FOR INDIVIDUAL EXPERIENCE //
sort ID JobN
qui: by ID: gen MiddlewareExp = sum(MIDDLE)

// CONSTRUCT VARIABLES THAT CAPTURE FIRST AND LAST PRODUCTS DEVELOPER HAS RELEASED // 

* Publisher Level *
sort publisher urlnew ID
qui: by publisher urlnew: gen temp = 1 if _n == 1
qui: by publisher: gen RunningNPubTitles = sum(temp)
qui: by publisher: egen TotalNPubTitles = sum(temp)

qui: gen FirstPublisherRelease = 1 if RunningNPubTitles == 1
qui: replace FirstPublisherRelease = 0 if FirstPublisherRelease == .
qui: gen FinalPublisherRelease = 1 if RunningNPubTitles == TotalNPubTitles
qui: replace FinalPublisherRelease = 0 if FinalPublisherRelease == .
qui: drop temp

* Developer Level *
sort developer urlnew developer_id
qui: by developer urlnew:  gen temp = 1 if _n == 1
qui: by developer: gen RunningNDevTitles = sum(temp)
qui: by developer: egen MaxDevNTitles = sum(temp)
qui: drop temp

qui: gen FirstDeveloperRelease = 1 if RunningNDevTitles == 1
qui: replace FirstDeveloperRelease = 0 if FirstDeveloperRelease == .
qui: gen LastDeveloperRelease = 1 if RunningNDevTitles == MaxDevNTitles
qui: replace LastDeveloperRelease = 0 if LastDeveloperRelease == .

// CONSTRUCT VARIABLES FOR PROJECT SIZE AND IDENTIFIER PER TITLE //
sort urlnew date
qui: bysort urlnew: gen ProjectSize = _N
qui: bysort urlnew: gen ProjectIdentifier = 1 if _n == 1

// CONSTRUCT INDICATOR VARIABLE FOR PERIOD AFTER 2002 (WHEN MIDDLEWARE BECAME MORE COMMON) //
qui: gen Post2002 = 1 if release_year >= 2002
qui: replace Post2002 = 0 if Post2002 == .

// CONSTRUCT VARIABLE FOR DIFFUSION OF MIDDLEWARE WITHIN GENRE //
gsort urlnew -npd_supergenre
*qui: by urlnew: replace npd_supergenre=npd_supergenre[_n-1] if npd_supergenre[_n-1]!=""
qui: bysort SHOOTER release_year: egen temp_0 = mean(MIDDLE) if ProjectIdentifier == 1
qui: bysort SHOOTER release_year: egen ShareMiddleware = max(temp_0)

// EXPERIENCE WITH COMPANY //
sort ID developer JobN
qui: by ID developer: gen N_Projects_Dev = _n - 1

In [11]:
// CREATE NEW VARIABLES WITH NAMES MORE EASY TO USE IN REGRESSION //

qui: gen Creative = 1 if HID == 3
qui: replace Creative = 0 if HID == 2

qui: gen Role = "Creative" if HID == 3
qui: replace Role = "Programmer" if HID == 2

qui: gen POST = Post2002
qui: gen PopMIDDLE = PopularMiddleware
qui: gen ShareMIDDLE = ShareMiddleware

## 3.0 Output Data for Analysis
-----


In [12]:
save "./Data/MISQ - 2025 - Data for Analysis.dta", replace

file ./Data/MISQ - 2025 - Data for Analysis.dta saved
